Espaço de estados é definido por uma grelha de células booleanas de dimensão N * N (com N > 16), estas N^2 são referidas como "normais"
Inicialmente todas as células normais estão mortas menos as células 1 <= i,j <= 3, que estão vivas. Dá-se o estado de erro quando todas as células normais estão mortas.

Uma vez que a grelha devia de ser infinita, implementamos umas regras para que seja finito:
- (células iniciais) 2N + 1 corresponde a um dos indices, i ou j tem de ser 0. Estas células tem valores constantes, que vão ser gerados aleatoriamente com um probabilidade 1/2 de estarem vivas.
- (células finais) 2N + 3 corresponde a um dos indices, i ou j tem que ter o valor de N + 1. As células finais tem o valor constante de zero (estão mortas)

Regra "B3|S23"
- Nasce se tiver EXATAMENTE 3 vizinhos vivos
- Sobrevie se tiver 2 ou 3 vizinhos vivos
    Caso contrário morre

As células (i0, j0), (i1, j1) são vizinhas se e só se max(|i0 - i1|, |j0 - j1|) = 1

In [2]:
from pysmt.shortcuts import *
import random

#Podemos mudar para gerar outra grelha inicial
seed = 4 

def declare(inst, N):
    estado = {}
    estado['grelha'] = {
        (i, j): Symbol(f"C_{inst}_{i}_{j}", BOOL)
        for i in range(N + 2) for j in range(N + 2)
    }
    return estado

def init(estado, N):
    rng = random.Random(seed)
    restricoes = []
    for (i,j), c in estado['grelha'].items():
        if i == 0 or j == 0:
            valor_c = rng.randint(0,1) == 1
        elif i == N+1 or j == N+1:
            valor_c = False
        else:
            valor_c = True if i <= 3 and j <= 3 else False
        restricoes.append(Iff(c, Bool(valor_c)))
    return And(restricoes)


def conta_vizinhos(atual, i, j):
    vizinhos = []
    for x in range(i-1, i+2):
        for y in range(j-1, j+2):
            if (x,y) != (i,j):
                vizinhos.append(Ite(atual['grelha'][(x,y)], Int(1), Int(0)))
    return Plus(vizinhos)

def transicoes(atual, proximo, N):
    restricoes = []
    #bordas
    for ij in range(N+2):
        restricoes.append(Iff(atual['grelha'][(ij,0)],proximo['grelha'][(ij,0)]))
        restricoes.append(Iff(atual['grelha'][(0,ij)],proximo['grelha'][(0,ij)]))
        restricoes.append(Iff(atual['grelha'][(ij,N+1)],proximo['grelha'][(ij,N+1)]))
        restricoes.append(Iff(atual['grelha'][(N+1,ij)],proximo['grelha'][(N+1,ij)]))
    
    for i in range(1, N+1):
        for j in range(1, N+1):
            vizinhos_vivos = conta_vizinhos(atual, i, j)
            restricoes.append(Iff(proximo['grelha'][(i,j)], Or(Equals(vizinhos_vivos, Int(3)), And(atual['grelha'][(i,j)], Equals(vizinhos_vivos, Int(2))))))
    return And(restricoes)

def transicoes_Uma(atual, proximo, N):
    restricoes = []
    #bordas
    for ij in range(N+2):
        restricoes.append(Iff(atual['grelha'][(ij,0)],proximo['grelha'][(ij,0)]))
        restricoes.append(Iff(atual['grelha'][(0,ij)],proximo['grelha'][(0,ij)]))
        restricoes.append(Iff(atual['grelha'][(ij,N+1)],proximo['grelha'][(ij,N+1)]))
        restricoes.append(Iff(atual['grelha'][(N+1,ij)],proximo['grelha'][(N+1,ij)]))

    escolha = {(i, j): Symbol(f"esc_{t}_{i}_{j}", BOOL)
               for i in range(1, N+1) for j in range(1, N+1)}
    restricoes.append(ExactlyOne(escolha.values()))

    
    for i in range(1, N+1):
        for j in range(1, N+1):
            vizinhos_vivos = conta_vizinhos(atual, i, j)
            restricoes.append(
                Ite(escolha[(i,j)],
                    Iff(proximo['grelha'][(i,j)],
                        Or(Equals(vizinhos_vivos, Int(3)), And(atual['grelha'][(i,j)], Equals(vizinhos_vivos, Int(2))))
                       ),
                    Iff(proximo['grelha'][(i,j)], atual['grelha'][(i,j)])
                   )
            )
    return And(restricoes)

def modelo(declare, init, trans, k, N, prop):
    with Solver(name="z3") as s:
        estados = [declare(inst, N) for inst in range(k + 1)]

        restricoes = []
        formula_inicial = init(estados[0], N)
        restricoes.append(formula_inicial)

        # Para todos os outros tem que ser uma transição valida para todos os outros
        for i in range(k):
            restricoes.append(trans(estados[i], estados[i + 1], N))

        
        restricoes.append(Or(prop(estados, N)))

        s.add_assertion(And(restricoes))


        #Encontramos um estado de erro
        if s.solve():
            model = s.get_model()
            print(model)
        else:
            print(f"Não foi encontrado nenhum estado de erro com {k} iterações")


# Estado de erro, todas as células normais estarem mortas
def erro(estado, N):
    restricoes = []
    for i in range(1, N+1):
        for j in range(1, N+1):
            restricoes.append(Iff(estado['grelha'][(i,j)], Bool(False)))
    return And(restricoes)

def alcanca_erro(estados, N):
    return Or([erro(estado, N) for estado in estados])

def verificar_estados(estados, vivo, k, i, j):
    celulas = [estado['grelha'][(i,j)] for estado in estados]
    return And(celulas) if vivo else And([Not(celula) for celula in celulas])

def muda_de_estado(estados, N):
    celulas = []
    for i in range(1, N+1):
        for j in range(1, N+1):
            celulas.append(Or(verificar_estados(estados, True, k, i, j), verificar_estados(estados, False, k, i, j)))

    return Or(celulas)

def estados_iguais(e1, e2, N):
    iguais = []
    for i in range(1, N+1):
        for j in range(1, N+1):
            iguais.append(Iff(e1['grelha'][(i,j)], e2['grelha'][(i,j)]))
    return And(iguais)

def fecha_laco(estados, N):
    k = len(estados) - 1
    return Or([estados_iguais(estados[k], estados[l], N) for l in range(k)])

def celula_permanente(estados, N):
    return And(fecha_laco(estados, N), muda_de_estado(estados, N))
        

#Sabemos que temos que ter um N > 16
N = 17
#Vamos ter duas transições, mas podemos ter as que quisermos
#neste caso, vamos ter 3 estados, exemplo do porquê E0 -- T --> E1 -- T --> E2, mas vai ser sempre k + 1
k = 2
modelo(declare,init,transicoes,k, N, celula_permanente)

Não foi encontrado nenhum estado de erro com 2 iterações
